# 🍎 Chat Completions with AIProjectClient 🍏

In this notebook, we'll demonstrate how to perform **Chat Completions** using the **Azure AI Foundry** SDK. We'll combine **`azure-ai-projects`**

1. **Initialize** an `AIProjectClient`.
2. **Obtain** a `get_openai_clients` client to do direct LLM calls.
3. **Use** a **prompt template** to add system context.
4. **Send** user prompts in a health & fitness theme.


In [ ]:
import os
import json
from pathlib import Path

def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

start_dir = os.getcwd()  # Use current directory
file_path = find_cred_json(start_dir)

print(f"Found cred.json at: {file_path}")

try:
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)
    
    print("Project Connection String:", loaded_config['PROJECT_CONNECTION_STRING'])
    print("Tenant ID:", loaded_config['TENANT_ID'])
    print("Model Deployment ID:", loaded_config['MODEL_DEPLOYMENT_NAME'])

except FileNotFoundError:
    print(f"Could not find file at: {file_path}")
except json.JSONDecodeError:
    print(f"File exists but contains invalid JSON")
except TypeError:
    print("File path was None — cred.json not found in any parent directories.")


## 1. Initial Setup
Load environment variables, create an `AIProjectClient`, and fetch a `ChatCompletionsClient`. We'll also define a **prompt template** to show how you might structure a system message.


In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Initialize your project client (make sure PROJECT_ENDPOINT is set in env vars)
project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=loaded_config["PROJECT_ENDPOINT"],
)

# Get authenticated Azure OpenAI client
openai_client = project_client.get_openai_client(
    api_version="2024-10-21"  # Use correct API version for your environment
)


### Prompt Template
We'll define a quick **system** message that sets the context as a friendly, disclaimer-providing fitness assistant.

```txt
SYSTEM PROMPT (template):
You are FitChat GPT, a helpful fitness assistant.
Always remind users: I'm not a medical professional.
Be friendly, provide general advice.
...
```

We'll then pass user content as a **user** message.


In [ ]:
def chat_with_fitness_assistant(user_input: str) -> str:
    """Use chat completions to get a response from our LLM."""
    system_text = (
        "You are FitChat GPT, a friendly fitness assistant.\n"
        "Always remind users: I'm not a medical professional.\n"
        "Answer with empathy and disclaimers."
    )

    # No more SystemMessage/UserMessage — just plain dicts
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_input},
    ]

    # Send the request
    response = openai_client.chat.completions.create(
        model=loaded_config["MODEL_DEPLOYMENT_NAME"],  # Set in your env
        messages=messages
    )

    return response.choices[0].message.content

print("chat_with_fitness_assistant updated and ready for latest SDK!")

## 2. Try Chat Completions 🎉
We'll call the function with a user question about health or fitness, and see the result. Feel free to modify the question or run multiple times!


In [ ]:
user_question = "How can I start a beginner workout routine at home?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ User:", user_question)
print("🤖 Assistant:", reply)

## 3. Another Example: Prompt Template with Fill-Ins 📝
We can go a bit further and add placeholders in the system message. For instance, imagine we have a **userName** or **goal**. We'll show a minimal example.


In [ ]:
def chat_with_template(user_input: str, user_name: str, goal: str) -> str:
    """Chat with a system prompt dynamically customized for the user."""
    # Construct and fill system template
    system_prompt = (
        f"You are FitChat GPT, an AI personal trainer for {user_name}.\n"
        f"Your user wants to achieve: {goal}.\n"
        "Remind them you're not a medical professional. Offer friendly advice."
    )

    # Prepare messages in OpenAI API format
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]

    # Send the request to the LLM
    response = openai_client.chat.completions.create(
        model=loaded_config["MODEL_DEPLOYMENT_NAME"],  # Set this in env vars
        messages=messages
    )

    return response.choices[0].message.content

# Example usage
templated_user_input = "What kind of home exercise do you recommend for a busy schedule?"
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="Jordan",
    goal="increase muscle tone and endurance"
)

print("🗣️ User:", templated_user_input)
print("🤖 Assistant:", assistant_reply)


## 🎉 Congratulations!
You've successfully performed **chat completions** with the Azure AI Foundry's `AIProjectClient` and `azure-ai-inference`. You've also seen how to incorporate **prompt templates** to tailor your system instructions.
